In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import numpy as np
import rasterio
from tqdm.notebook import tqdm
import pandas as pd
import rasterio
import random
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as v2
import torchvision.models as models
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchinfo import summary

from terratorch.models.pixel_wise_model import freeze_module
from huggingface_hub import hf_hub_download
from terratorch.models.backbones.prithvi_mae import PrithviViT

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed) # Set seed for Python's built-in random number generator
    np.random.seed(seed) # Set seed for numpy
    if torch.cuda.is_available(): # Set seed for CUDA if available
        torch.cuda.manual_seed_all(seed)
        # Set cuDNN's random number generator seed for deterministic behavior
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [ ]:
batch_size = 32
num_workers = 6
pa_presence_threshold = 1
num_classes_total = 11255
landsat_year_len = 18
bioclim_month_len = landsat_year_len*12-1
validation_prop = 0.1

mean_landsat = 1*np.array([ 15.0698,   16.0923,    7.9312,   68.9794,   47.9505,   24.8804, 7089.4349, 2830.6658])
std_landsat =  1*np.array([ 11.7218,   10.2417,    9.6499,   18.7112,   13.1681,    9.2436, 3332.3618,   56.7270])
mean_sentinel = 1*np.array([ 624.8547,  684.7646,  456.7674, 2924.1753])
std_sentinel =  1*np.array([ 416.0408,  351.1005,  315.8956,  943.6141])

class HorizontalCycleTransform(torch.nn.Module):
    def forward(self, img):
        img2 = torch.cat([img, img], -1)
        start = torch.randint(img.shape[-1], (1,))[0]
        new_img = img2[:,:,start:start+img.shape[-1]]
        return new_img

# class HorizontalCycleTransform(torch.nn.Module):
#     def forward(self, img):
#         new_img = img[:,:,torch.randperm(img.shape[-1])]
#         return new_img

transform_landsat = v2.Compose([
    v2.Normalize(mean_landsat, std_landsat),
    #v2.RandomHorizontalFlip(p=0.5),
    HorizontalCycleTransform()
])
transform_landsat_test = v2.Compose([
    v2.Normalize(mean_landsat, std_landsat),
])
transform_sentinel = v2.Compose([
    v2.Normalize(mean_sentinel, std_sentinel),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(180),
    v2.RandomResizedCrop(size=64, scale=(0.25, 1.0))
])
transform_sentinel_test = v2.Compose([
    v2.Normalize(mean_sentinel, std_sentinel)
])

In [ ]:
set_seed(42)
path_data = os.getenv('GLC_DATA_PATH')
train_path_sentinel = os.path.join(path_data, "SatelitePatches/PA-train")
train_path_landsat = os.path.join(path_data, "SateliteTimeSeries-Landsat/cubes/PA-train")
train_path_bioclim = os.path.join(path_data, "BioclimTimeSeries/cubes/PA-train")
train_metadata = pd.read_csv(os.path.join(path_data, "GLC25_PA_metadata_train.csv"))
train_metadata = train_metadata.dropna(subset="speciesId").reset_index(drop=True)
train_metadata['speciesId'] = train_metadata['speciesId'].astype(int)
train_metadata["speciesIdOrig"] = train_metadata['speciesId']
tmp = train_metadata["speciesId"].value_counts() >= pa_presence_threshold
train_metadata.loc[~train_metadata["speciesId"].isin(tmp[tmp].index), "speciesId"] = -1
sp_categorical = train_metadata["speciesId"].astype("category").values
num_classes = len(sp_categorical.categories)
train_metadata['speciesId'] = sp_categorical.codes

tmp = train_metadata.groupby("surveyId").agg({"surveyId":"first", "lat":"first", "lon":"first", "areaInM2":lambda x: list(x.unique()), "region":"first", "country":"first", "speciesId":list})
train_label_series = tmp.set_index("surveyId").speciesId
train_metadata = tmp.drop(columns=["speciesId"]).set_index("surveyId", drop=False)
train_metadata["area"] = train_metadata["areaInM2"].apply(lambda x: 1.0 if np.isinf(x).all() else np.mean(x, where=~np.isinf(x)))
train_metadata["areaLog"] = np.log10(train_metadata["area"])

train_metadata['area'].fillna(train_metadata['area'].mean(), inplace=True)
train_metadata['areaLog'].fillna(train_metadata['areaLog'].mean(), inplace=True)
train_metadata["conFra"] = train_metadata["country"] == "France"
train_metadata["conDen"] = train_metadata["country"] == "Denmark"
train_metadata["conNet"] = train_metadata["country"] == "Netherlands"
train_metadata["conIta"] = train_metadata["country"] == "Italy"
train_metadata["conAus"] = train_metadata["country"] == "Austria"
train_metadata["conOther"] = ~train_metadata["country"].isin(["France","Denmark","Netherlands","Italy","Austria"])
train_elevation = pd.read_csv(os.path.join(path_data, "EnvironmentalValues", "Elevation", "GLC25-PA-train-elevation.csv"), index_col=0)
train_elevation['Elevation'].fillna((train_elevation['Elevation'].mean()), inplace=True)
train_soil = pd.read_csv(os.path.join(path_data, "EnvironmentalValues", "SoilGrids", "GLC25-PA-train-soilgrids.csv"), index_col=0)
for column in train_soil.columns: train_soil[column].fillna((train_soil[column].mean()), inplace=True)
train_worldcover = pd.read_csv(os.path.join(path_data, "worldcover", "s2_pa_train_survey_points_with_worldcover.csv"), index_col=0)
train_wcdummy = pd.get_dummies(train_worldcover["class"], prefix="wc")
train_wcdummy.drop(columns="wc_70", inplace=True)
train_wcdummy.drop(columns="wc_100", inplace=True)
train_landcover = pd.read_csv(os.path.join(path_data, "EnvironmentalValues", "LandCover", "GLC25-PA-train-landcover.csv"), index_col=0)
landcover_col_ind = [0,2,3,5,8,11,12]
train_landcover = train_landcover.iloc[:, landcover_col_ind]

print("All rows match: ", (train_metadata.index==train_elevation.index).all() and (train_metadata.index==train_soil.index).all() \
     and (train_metadata.index==train_worldcover.index).all() and (train_metadata.index==train_landcover.index).all())
cov_columns = ["areaLog", "Elevation", "conFra", "conDen", "conNet", "conIta", "conAus", "conOther"] + list(train_soil.columns) + list(train_wcdummy.columns) + list(train_landcover.columns)
train_combined = pd.concat([train_metadata, train_elevation.Elevation, train_soil, train_wcdummy, train_landcover], axis=1)
cov_norm_coef = train_combined.loc[:,cov_columns].agg(['mean', 'std'])
dummy_columns = ["conFra","conDen","conNet","conIta","conAus","conOther"] + list(train_wcdummy.columns)
cov_norm_coef.loc["mean",dummy_columns] = 0
cov_norm_coef.loc["std",dummy_columns] = 1
train_combined.loc[:,cov_columns] = (train_combined.loc[:,cov_columns] - cov_norm_coef.loc["mean"]) / cov_norm_coef.loc["std"]

val_ind = np.sort(train_combined.surveyId.sample(frac=validation_prop).values)
train_data, val_data = [x.reset_index(drop=True) for _, x in train_combined.groupby(train_combined.surveyId.isin(val_ind))]
train_label_dict = train_label_series[train_data.surveyId].to_dict()
val_label_dict = train_label_series[val_data.surveyId].to_dict()


In [ ]:
# Load Test metadata
test_path_sentinel = os.path.join(path_data, "SatelitePatches/PA-test")
test_path_landsat = os.path.join(path_data, "SateliteTimeSeries-Landsat/cubes/PA-test")
test_path_bioclim = os.path.join(path_data, "BioclimTimeSeries/cubes/PA-test")
test_metadata = pd.read_csv(os.path.join(path_data, "GLC25_PA_metadata_test.csv")).set_index("surveyId", drop=False).sort_index()
test_metadata.rename(columns={"areaInM2": "area"}, inplace=True)
test_metadata.replace({"area": [np.inf, -np.inf]}, 1.0, inplace=True)
test_metadata['areaLog'] = np.log10(test_metadata['area'])
test_metadata['area'].fillna(test_metadata['area'].mean(), inplace=True)
test_metadata['areaLog'].fillna(test_metadata['areaLog'].mean(), inplace=True)
test_metadata["conFra"] = test_metadata["country"] == "France"
test_metadata["conDen"] = test_metadata["country"] == "Denmark"
test_metadata["conNet"] = test_metadata["country"] == "Netherlands"
test_metadata["conIta"] = test_metadata["country"] == "Italy"
test_metadata["conAus"] = test_metadata["country"] == "Austria"
test_metadata["conOther"] = ~test_metadata["country"].isin(["France","Denmark","Netherlands","Italy","Austria"])
test_elevation = pd.read_csv(os.path.join(path_data, "EnvironmentalValues", "Elevation", "GLC25-PA-test-elevation.csv"), index_col=0).sort_index()
test_elevation = test_elevation.loc[test_elevation.index.isin(test_metadata.index)]
test_elevation['Elevation'].fillna((test_elevation['Elevation'].mean()), inplace=True)
test_soil = pd.read_csv(os.path.join(path_data, "EnvironmentalValues", "SoilGrids", "GLC25-PA-test-soilgrids.csv"), index_col=0).sort_index()
test_soil = test_soil.loc[test_soil.index.isin(test_metadata.index)]
for column in test_soil.columns: test_soil[column].fillna((test_soil[column].mean()), inplace=True)
test_worldcover = pd.read_csv(os.path.join(path_data, "worldcover", "pa_test_survey_points_with_worldcover.csv"), index_col=0).sort_index()
test_wcdummy = pd.get_dummies(test_worldcover["class"], prefix="wc")
test_wcdummy.drop(columns="wc_100", inplace=True)
# test_wcdummy.insert(6, "wc_70", False)
test_landcover = pd.read_csv(os.path.join(path_data, "EnvironmentalValues", "LandCover", "GLC25-PA-test-landcover.csv"), index_col=0).sort_index()
test_landcover = test_landcover.loc[test_landcover.index.isin(test_metadata.index)]
test_landcover = test_landcover.iloc[:, landcover_col_ind]

print("All surveyId match: ", (test_metadata.index==test_elevation.index).all() and (test_metadata.index==test_soil.index).all() \
     and (test_metadata.index==test_worldcover.index).all() and (test_metadata.index==test_landcover.index).all())
test_combined = pd.concat([test_metadata, test_elevation.Elevation, test_soil, test_wcdummy, test_landcover], axis=1)
test_combined.loc[:,cov_columns] = (test_combined.loc[:,cov_columns] - cov_norm_coef.loc["mean"]) / cov_norm_coef.loc["std"]
test_combined.reset_index(drop=True, inplace=True)


In [ ]:
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from config import SCRATCH_PATH
pred_path = os.path.join(SCRATCH_PATH, 'prithvi/0424_225841_addlandcover_swapDOorder_testfail/0425_012850_e075_vloss0.008629_vf0.3079.csv')
pred = pd.read_csv(pred_path, index_col=0)
pred = pd.concat([test_combined.set_index("surveyId"), pred], axis=1)
pred_ok = pred.loc[~pred.predictions.isna()]
pred_na = pred.loc[pred.predictions.isna()]

In [ ]:
tmp = pred_na.describe()
tmp.shape

In [ ]:
with pd.option_context('display.max_rows', 20, 'display.max_columns', None): 
    display(pd.concat([pred_na.describe(), 0*pred_na.describe().iloc[:1], pred_ok.describe()]))

In [ ]:
with pd.option_context('display.max_rows', 10, 'display.max_columns', None): 
    display(train_combined.describe())

In [ ]:
test_landcover.describe()

In [ ]:
train_landcover.describe()

In [ ]:
test_landcover

In [ ]:
test_landcover.tail(30)

In [ ]:
test_landcover[test_landcover.index>=5000000]

In [ ]:
test_landcover = pd.read_csv(os.path.join(path_data, "EnvironmentalValues", "LandCover", "GLC25-PA-test-landcover.csv"), index_col=0).sort_index()
test_landcover = test_landcover.loc[test_landcover.index.isin(test_metadata.index)]
test_landcover.loc[test_landcover.nunique(1) <= 1]

In [ ]:
test_landcover = pd.read_csv(os.path.join(path_data, "EnvironmentalValues", "LandCover", "GLC25-PA-test-landcover.csv"), index_col=0).sort_index()
test_landcover.loc[test_landcover.nunique(1) <= 1]

In [ ]:
train_landcover = pd.read_csv(os.path.join(path_data, "EnvironmentalValues", "LandCover", "GLC25-PA-train-landcover.csv"), index_col=0)
train_landcover.loc[train_landcover.nunique(1) <= 1].index

In [ ]:
test_landcover

In [ ]:
10114 / 14829